## Week 8: Importing and Cleaning Data

**Readings**
* [pandas: IO tools (`read_csv`, `read_excel`, ...)](https://pandas.pydata.org/docs/user_guide/io.html)
* [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
* [Python for Data Analysis, ch. 7: Data Cleaning and Preparation](https://wesmckinney.com/book/data-cleaning.html)
* [Python Data Science Handbook: Handling Missing Data](https://jakevdp.github.io/PythonDataScienceHandbook/03.04-missing-values.html)

**Today's question:** real public data almost never loads cleanly. We work with three real files and fix what is wrong with them, keeping a written record of every decision. You will need exactly this record for your final project.

| Dataset | Publisher | One row is | Why it is useful today |
|---|---|---|---|
| [2022 presidential election, first round, by département](https://www.data.gouv.fr/datasets/election-presidentielle-des-10-et-24-avril-2022-resultats-definitifs-du-1er-tour/) | Ministère de l'Intérieur, data.gouv.fr | a département | encoding, separators, French decimals, a broken header |
| [Répertoire national des élus: mayors and deputies](https://www.data.gouv.fr/datasets/repertoire-national-des-elus-1/) | Ministère de l'Intérieur, data.gouv.fr | an elected official | missing values, dates, codes, duplicates and keys |
| [Women in national parliaments](https://data.worldbank.org/indicator/SG.GEN.PARL.ZS) | World Bank (from the Inter-Parliamentary Union) | a country **or a group of countries** | junk header lines, rows that are not countries, missing years |

The files were downloaded on 2 November 2026 and are stored in `data/`, so results do not change if the publisher updates them.

In [ ]:
import os
import urllib.request

import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)
print(pd.__version__)

# If you opened this notebook in Colab, the data/ folder is not there yet: this cell fetches it.
BASE_URL = "https://raw.githubusercontent.com/cg67qowo/Introduction-to-Python/main/Week08-ImportingCleaning/data/"
FILES = [
    "resultats_t1_departements_2022.txt",
    "resultats_t1_departements_2022.xlsx",
    "elus_maires_2026.csv",
    "elus_deputes_2026.csv",
    "wb_women_parliament.csv",
    "wb_country_metadata.csv",
    "departements.json",
]
os.makedirs("data", exist_ok=True)
for name in FILES:
    if not os.path.exists("data/" + name):
        urllib.request.urlretrieve(BASE_URL + name, "data/" + name)
print(sorted(os.listdir("data")))

### 8.0 Warm-up: predict the output

Run each cell only **after** you have written down what you expect it to print.

In [ ]:
turnout = pd.Series([77.7, None, 76.6])
print(len(turnout))
print(turnout.count())
print(turnout.mean())

In [ ]:
code = pd.Series(["01", "02", "2A"])
print(code + "000")
print(int("01") + 1)

In [ ]:
parties = pd.Series(["PS", "PS", "RN", "ps"])
print(parties.nunique())
print(parties.duplicated().sum())

**What to notice.** `len` counts rows while `count` counts non-missing values, and `mean` silently skips missing values. A code such as `"01"` is text: as a number it would lose its zero, and `"2A"` could not be a number at all. To Python, `"PS"` and `"ps"` are two different parties.

### 8.1 Importing a real CSV: the 2022 election results

We want a table with one row per département. Let's try the obvious thing first.

In [ ]:
path = "data/resultats_t1_departements_2022.txt"

try:
    pd.read_csv(path)
except UnicodeDecodeError as error:
    print("UnicodeDecodeError:", error)

**Problem 1: encoding.** By default pandas expects UTF-8, and this file was saved by older software in `latin-1`, where `é` is a different byte. If a file fails with `UnicodeDecodeError`, try `encoding="latin-1"`.

In [ ]:
attempt = pd.read_csv(path, encoding="latin-1")
print(attempt.shape)
print(attempt.columns[0][:70])

No error, but only **one column**: everything is glued together. A correct-looking load is not a correct load. When in doubt, look at the raw text before pandas touches it.

In [ ]:
with open(path, encoding="latin-1") as f:
    header = f.readline()
    first_row = f.readline()

print(header[:110])
print(first_row[:110])
print("fields in the header:", header.count(";") + 1)
print("fields in the first row:", first_row.count(";") + 1)

The raw text reveals three things:

1. **Separator:** values are separated by `;`, the usual choice in French files, because `,` is the decimal mark.
2. **Decimals:** `22,26` uses a French decimal comma, so we need `decimal=","`.
3. **The header is broken:** it names 23 columns while each row has 89 fields. The first candidate has named columns; the other 11 candidates have none.

What happens if we fix only the separator? No error again, but look at the first column.

In [ ]:
attempt = pd.read_csv(path, encoding="latin-1", sep=";")
print(attempt.shape)
print(attempt["Code du département"].head(3).tolist())
print(attempt["Inscrits"].head(3).tolist())

**The most dangerous kind of error is the silent one.** pandas saw more fields than column names, moved the extra leading fields into the row index, and aligned the 23 column names with the **last** 23 fields. The shape looks fine, while `"Code du département"` now holds the name of a candidate (JADOT) and `"Inscrits"` holds a percentage.

The fix is to write out all 89 column names ourselves, with a loop over the candidates in the order they appear in the file.

In [ ]:
base_columns = header.strip().split(";")[:17]
candidates = ["ARTHAUD", "ROUSSEL", "MACRON", "LASSALLE", "LE PEN", "ZEMMOUR",
              "MELENCHON", "HIDALGO", "JADOT", "PECRESSE", "POUTOU", "DUPONT-AIGNAN"]

column_names = list(base_columns)
for candidate in candidates:
    for field in ["sexe", "nom", "prenom", "voix", "pct_ins", "pct_exp"]:
        column_names.append(field + "_" + candidate)

election = pd.read_csv(
    path,
    encoding="latin-1",                     # problem 1: not UTF-8
    sep=";",                                # problem 2: not commas
    decimal=",",                            # problem 3: 22,26 means 22.26
    names=column_names,                     # problem 4: incomplete header...
    header=0,                               # ...so replace the first line
    index_col=False,                        # never use a column as the index
    dtype={"Code du département": str},     # problem 5: "01" must keep its zero
)
print(election.shape)
print(election.iloc[:3, [0, 1, 3, 5]])

**Never trust an import until it matches something you can verify.** The official national result gave Emmanuel Macron 9,783,058 votes in the first round. Do our columns agree? Also check that each block really belongs to the candidate we think it does.

In [ ]:
print(election["voix_MACRON"].sum())
print(election["nom_MACRON"].unique().tolist())
print(election["nom_MELENCHON"].unique().tolist())

##### Exercise 1

Keep only the rows where the code does **not** start with `Z`. The `Z` codes are overseas territories and French citizens abroad. Print how many rows are left, then the 3 départements with the highest abstention (`"% Abs/Ins"`), showing only the name and the abstention rate.

Hint: `election["Code du département"].str.startswith("Z")` gives True/False, and `~` reverses it.

In [ ]:
# Write your solution here

Expected output:

```text
96
Libellé du département  % Abs/Ins
           Haute-Corse      37.38
          Corse-du-Sud      37.19
     Seine-Saint-Denis      30.22
```

##### Exercise 2

Compute Marine Le Pen's national share of valid votes as `voix_LE PEN / Exprimés`, using the sums over **all** 107 rows. Print it as a percentage rounded to 2 decimals. The official figure is 23.15%: if yours differs, the import is wrong.

In [ ]:
# Write your solution here

Expected output:

```text
23.15
```

### 8.2 Other formats and messy headers

The same steps work for other sources. Change the function, and keep checking the result.

| Source | Function |
|---|---|
| CSV or text file | `pd.read_csv(path, sep=..., encoding=...)` |
| Excel | `pd.read_excel(path, sheet_name=...)` |
| JSON (often from a web API) | `pd.read_json(path)` |
| A file on the web | the same functions with a URL instead of a path, e.g. `pd.read_csv("https://...")` |

The ministry also publishes the election file as Excel. Encoding and decimals are no longer a problem, because Excel stores real numbers, but the missing headers are still there.

In [ ]:
# read_excel needs the openpyxl package. If it is missing: pip install openpyxl
excel = pd.read_excel("data/resultats_t1_departements_2022.xlsx",
                      dtype={"Code du département": str})
print(excel.shape)
print(excel["Code du département"].iloc[0], excel["Libellé du département"].iloc[0], excel["% Abs/Ins"].iloc[0])
print(excel.columns[20:27].tolist())

JSON is what most web APIs return. This file comes from the French government's geographic API (`https://geo.api.gouv.fr/departements?fields=nom,code,codeRegion`).

In [ ]:
geo = pd.read_json("data/departements.json", dtype={"code": str, "codeRegion": str})
print(geo.shape)
print(geo.head(3))

Now the World Bank file. Its first lines are not data, a common sight in files from international organisations.

In [ ]:
wb_path = "data/wb_women_parliament.csv"

with open(wb_path, encoding="utf-8-sig") as f:
    for i in range(5):
        print(i, repr(f.readline()[:75]))

try:
    pd.read_csv(wb_path)
except pd.errors.ParserError as error:
    print("ParserError:", error)

In [ ]:
wb = pd.read_csv(wb_path, skiprows=4)    # the real header is on line 4, counting from 0
print(wb.shape)
print(wb.columns[:5].tolist(), "...", wb.columns[-3:].tolist())
print(wb["Unnamed: 70"].notna().sum())

Every line in the file ends with a comma, so pandas invents an empty last column, `Unnamed: 70`, with 0 values in it. Remove columns that are entirely empty with `dropna(axis=1, how="all")`.

In [ ]:
wb = wb.dropna(axis=1, how="all")
print(wb.shape)
print(wb.columns[-2:].tolist())

##### Exercise 3

Read `data/wb_country_metadata.csv` into a DataFrame called `meta` (no special options are needed). Print its shape, its column names, and the first 2 rows of `"TableName"` and `"Region"`. Then drop any column that is entirely empty and print the new shape.

In [ ]:
# Write your solution here

Expected output:

```text
(265, 6)
['Country Code', 'Region', 'IncomeGroup', 'SpecialNotes', 'TableName', 'Unnamed: 5']
                     TableName                     Region
0                        Aruba  Latin America & Caribbean
1  Africa Eastern and Southern                        NaN
(265, 5)
```

### 8.3 Inspecting: the checklist to run on every new dataset

Next dataset: all 34,826 mayors in France, published a few months after the March 2026 municipal elections.

**A note on personal data.** This is an official public register, but it does contain the names and birth dates of real people. You do not need names to study a population, so we drop them immediately and never print individuals. This is good practice for your project too.

In [ ]:
mayors_raw = pd.read_csv("data/elus_maires_2026.csv", sep=";")
mayors = mayors_raw.drop(columns=["Nom de l'élu", "Prénom de l'élu"])

mayors = mayors.rename(columns={
    "Code du département": "dept_code",
    "Libellé du département": "dept",
    "Code de la collectivité à statut particulier": "coll_code",
    "Libellé de la collectivité à statut particulier": "coll",
    "Code de la commune": "commune_code",
    "Libellé de la commune": "commune",
    "Code sexe": "sex",
    "Date de naissance": "birth_date",
    "Code de la catégorie socio-professionnelle": "job_code",
    "Libellé de la catégorie socio-professionnelle": "job",
    "Date de début du mandat": "mandate_start",
    "Date de début de la fonction": "function_start",
})
print(mayors.shape)     # 1. How many rows and columns? What is one row?
mayors.head(3)          # 2. What does it look like?

In [ ]:
mayors.info()           # 3. Types and non-missing counts for each column

In [ ]:
mayors.describe()       # 4. Summary of numeric columns. Does each one make sense?

In [ ]:
print(mayors.nunique())                 # 5. How many distinct values in each column?
print(mayors["sex"].value_counts())     # 6. What are the categories?

**Read `describe()` critically.** The mean of `job_code` (about 50) means nothing, because this number is a label. `coll_code` looks numeric for the same bad reason. A numeric type does not make a column a quantity.

##### Exercise 4

Using `value_counts()`, print the 3 most common `job` labels among mayors. Then print the **percentage** of mayors for each value of `sex`, rounded to 1 decimal.

Hint: `value_counts(normalize=True)` returns proportions.

In [ ]:
# Write your solution here

Expected output:

```text
job
Ancien cadre                                      5281
Agriculteur sur moyenne exploitation              2426
Cadre administratif et commercial d'entreprise    2162
Name: count, dtype: int64
sex
M    77.1
F    22.9
Name: proportion, dtype: float64
```

### 8.4 Missing values: first understand, then decide

There is no universal rule such as "always drop". Ask **why** a value is missing, then write down what you did. Start by counting.

In [ ]:
missing = mayors.isna().sum()
print(missing[missing > 0])
print()
print((mayors.isna().mean() * 100).round(1)[missing > 0])

`coll_code` is empty for 98.6% of mayors. Should we panic, or drop the column? Let's look at who has a value, and who is missing a département.

In [ ]:
print(mayors["coll"].value_counts())
print()
no_dept = mayors[mayors["dept_code"].isna()]
print(no_dept["coll"].value_counts())

Nothing is lost here. **These values are missing by design:** most communes do not belong to a "collectivité à statut particulier", and the 139 communes in Martinique, Guyane, Polynésie française, Nouvelle-Calédonie and Saint-Pierre-et-Miquelon are recorded under a collectivité rather than a département. Filling them with an average, or dropping them, would be wrong (and would delete five territories from the analysis!). The correct fix uses the information in the neighbouring column.

In [ ]:
mayors["territory"] = mayors["dept"].fillna(mayors["coll"])
print(mayors["territory"].isna().sum())
print(mayors["territory"].nunique())

**Now a case where the choice really matters.** In the World Bank table, what is the world average share of women in parliament in 2025?

Before calculating anything, look at the rows: some of them are not countries.

In [ ]:
print(wb["Country Name"].iloc[[1, 3, 77, 259]].tolist())

meta = pd.read_csv("data/wb_country_metadata.csv")
is_real_country = meta["Region"].notna()      # groups of countries have no Region
country_codes = meta.loc[is_real_country, "Country Code"]

countries = wb[wb["Country Code"].isin(country_codes)].copy()
print(len(wb), "rows ->", len(countries), "countries and territories")
print("Average, all rows:", round(wb["2025"].mean(), 2))
print("Average, countries only:", round(countries["2025"].mean(), 2))

In [ ]:
missing_2025 = countries[countries["2025"].isna()]
print(len(missing_2025), "of", len(countries), "have no value for 2025:")
print(missing_2025["Country Name"].tolist())

**Is the missingness random? Clearly not.** The list mixes non-sovereign territories (Guam, Greenland) with countries whose parliament was dissolved or suspended (Afghanistan, Sudan, Kuwait). `dropna()` or the automatic skipping in `.mean()` therefore describes *countries with a functioning parliament*, a narrower population than "the world". Your written conclusion must say so.

The options:

| Option | Code | Use it when | Risk |
|---|---|---|---|
| Keep and report | `s.isna().sum()` | always the first step | none |
| Drop rows | `df.dropna(subset=["2025"])` | few are missing, and you say who | changes the population |
| Fill with a known value | `fillna(other_column)` | the true value exists elsewhere | none, if the source is right |
| Carry the last value forward | `df[years].ffill(axis=1)` | a slow-moving indicator; state the choice | may be out of date |
| Fill with the mean | `fillna(s.mean())` | **rarely**, and never without saying so | invents data, hides differences |

For example, two countries, Guinea-Bissau and Nepal, have a 2024 value but none for 2025. If we allow the latest value from the last decade, does our conclusion change?

In [ ]:
years = [str(y) for y in range(2015, 2026)]
latest = countries[years].ffill(axis=1)["2025"]
print("Countries with a value:", countries["2025"].notna().sum(), "->", latest.notna().sum())
print("Average, 2025 only:", round(countries["2025"].mean(), 2))
print("Average, latest value since 2015:", round(latest.mean(), 2))

The average moves by less than half a point. This kind of **robustness check** is simple to do and worth one line in your project: "the conclusion does not depend on this choice".

##### Exercise 5

Using `countries`, print the number of **missing** values in column `"2000"` and in column `"2025"`. Then print the average for each of those two years, rounded to 1 decimal. Can we compare these two averages directly? Answer in a comment.

In [ ]:
# Write your solution here

Expected output:

```text
2000 60 11.8
2025 33 26.2
```

##### Exercise 6

Make that fair comparison. Keep only the rows in `countries` with a value in **both** `"2000"` and `"2025"` (use `dropna(subset=[...])`). Print how many countries remain, and both averages rounded to 1 decimal.

In [ ]:
# Write your solution here

Expected output:

```text
152
11.9 26.9
```

### 8.5 Fixing data types

Three classic problems: numbers stored as text, identifiers stored as numbers, and dates stored as text.

**Numbers stored as text.** If we forget `decimal=","`, a column of percentages arrives as text. `mean()` then fails or gives nonsense, and `sort_values` puts `"9,5"` after `"22,1"`.

In [ ]:
as_text = pd.Series(["22,26", "27,06", "23,40", "n.d."])

as_numbers = pd.to_numeric(as_text.str.replace(",", "."), errors="coerce")
print(as_numbers)
print(as_numbers.mean())

`errors="coerce"` turns anything that cannot be converted, like `"n.d."` (non disponible), into `NaN`. Always count how many values it turned into `NaN`. If it is more than you expected, look at them before moving on.

**Identifiers stored as numbers.** Without instructions, pandas guesses the type of each column. Here is what can happen with a code column.

In [ ]:
codes = pd.Series(["01", "09", "75"])
print(codes.astype(int).tolist())       # the zeros are gone, and "01" no longer matches other files

print(mayors["commune_code"].head(2).tolist())
print(mayors["coll_code"].dropna().head(2).tolist())   # 94.0: a label should not be a decimal number

Read codes as text from the start: `pd.read_csv(path, dtype={"Code du département": str})`.

**Dates stored as text.** `info()` showed `birth_date` as text. Once converted, we can calculate with it. We fix a reference date, so the result does not change each time the notebook runs.

In [ ]:
mayors["birth_date"] = pd.to_datetime(mayors["birth_date"])
reference_date = pd.Timestamp("2026-11-02")
mayors["age"] = (reference_date - mayors["birth_date"]).dt.days / 365.25

print(mayors["birth_date"].dt.year.head(3).tolist())
print(mayors["age"].describe().round(1))

Check the range: minimum 20, maximum about 95. French mayors must be at least 18, so this is plausible, and no impossible values need handling. Run this kind of check on every numeric column.

**Careful with date formats.** This file uses `2026-03-15`. A French file might use `15/03/2026`, which requires `pd.to_datetime(s, dayfirst=True)`, or `format="%d/%m/%Y"` to be stricter. Otherwise 03/04 can silently become March 4th instead of April 3rd.

##### Exercise 7

Convert `mandate_start` to a date. Print the earliest and latest date. Then print how many mayors began their mandate **after** 22 March 2026, the second round of the municipal elections.

Hint: you can compare a date column with `pd.Timestamp("2026-03-22")`.

In [ ]:
# Write your solution here

Expected output:

```text
2026-03-15 2026-07-25
73
```

**Recoding categories.** `job` has 40 labels, too many to compare. The first digit of the INSEE code gives 8 broad groups, which we label using a dictionary from session 5 and `map`.

In [ ]:
job_groups = {
    1: "Farmers",
    2: "Craftspeople, shopkeepers, business owners",
    3: "Managers and higher professions",
    4: "Intermediate professions",
    5: "Employees",
    6: "Manual workers",
    7: "Retired",
    8: "Other, no professional activity",
}
mayors["job_group"] = (mayors["job_code"] // 10).map(job_groups)
print(mayors["job_group"].isna().sum())   # check: did every code find a label?
print(mayors["job_group"].value_counts())

##### Exercise 8

Create a column `is_woman` that is `True` when `sex` is `"F"`. Use `groupby` to print the share of women mayors (in %, 1 decimal) for each `job_group`, sorted from highest to lowest.

The average of a True/False column is the share of True values.

In [ ]:
# Write your solution here

Expected output:

```text
job_group
Employees                                     47.1
Other, no professional activity               46.6
Intermediate professions                      35.0
Managers and higher professions               23.5
Retired                                       17.7
Craftspeople, shopkeepers, business owners    15.3
Farmers                                       12.7
Manual workers                                12.1
```

### 8.6 Duplicates, keys and labels

**Exact duplicates** are rows repeated in full, often caused by appending a file twice. Here we check for them, and then simulate that accident to see the fix.

In [ ]:
print("Exact duplicates:", mayors.duplicated().sum())

# SIMULATED accident, not real data: the last 100 rows are appended twice
accident = pd.concat([mayors, mayors.tail(100)])
print(len(accident), "rows, duplicates:", accident.duplicated().sum())
print(len(accident.drop_duplicates()), "rows after drop_duplicates()")

**Key duplicates matter more.** Before anything else, ask: *what identifies one row?* Each commune has one mayor, so `commune_code` should be unique. What about the commune's name?

In [ ]:
print("Duplicated codes:", mayors["commune_code"].duplicated().sum())
print("Duplicated names:", mayors["commune"].duplicated().sum())
print(mayors["commune"].value_counts().head(3))

Of course: France has 12 communes called Sainte-Colombe. They are **different places**, and running `drop_duplicates("commune")` would delete 2,224 real mayors with no error message. **Names are not keys: use official codes whenever a dataset has them.**

**Same thing, different label.** Next week you will merge tables. Would the département names of our two files match?

In [ ]:
election_names = set(election["Libellé du département"])
mayor_names = set(mayors["dept"].dropna())
only_in_mayors = sorted(mayor_names - election_names)

print(len(only_in_mayors), "names only in the mayors file:")
print(only_in_mayors[:4])
print(sorted(election_names - mayor_names)[:4])

`Bouches-Du-Rhône` and `Bouches-du-Rhône`: the same place, but different text. A merge on the name would silently lose 21 départements. Two fixes: normalise the text, or, far better, use the code.

In [ ]:
def normalise(names):
    """Lower-case the names and remove spaces at each end."""
    return names.str.strip().str.lower()

print(sorted(set(normalise(mayors["dept"].dropna())) - set(normalise(election["Libellé du département"]))))
print(sorted(set(mayors["dept_code"].dropna()) - set(election["Code du département"])))
print(sorted(set(election["Code du département"]) - set(mayors["dept_code"].dropna())))

Normalising resolves all 21 name mismatches. But the codes reveal one more trap: **overseas territories have different codes in the two files**, e.g. Guadeloupe is `971` in one and `ZA` in the other. Even official codes need checking!

##### Exercise 9

Is `"Country Code"` a valid key for the `countries` table, and is `"Country Name"`? Print the number of duplicates for each. Then print what `normalise` from the cell above returns for `pd.Series(["  Sciences Po", "SCIENCES PO "])`, and how many unique values remain.

In [ ]:
# Write your solution here

Expected output:

```text
0
0
['sciences po', 'sciences po']
1
```

### 8.7 Putting it together: a reproducible cleaning function

A good notebook does not clean data through scattered edits. It goes from the raw file to the clean table in **one re-runnable step**, and it **records what happened**. Your final project needs the initial and final number of observations and an explanation of each exclusion.

In [ ]:
def clean_officials(path):
    """Load a Répertoire national des élus file and return a clean table."""
    raw = pd.read_csv(path, sep=";", dtype={"Code du département": str})
    log = [("raw file", len(raw))]

    # no names needed: keep personal data to the minimum
    df = raw.drop(columns=["Nom de l'élu", "Prénom de l'élu"])

    df = df.rename(columns={
        "Code du département": "dept_code",
        "Libellé du département": "dept",
        "Libellé de la collectivité à statut particulier": "coll",
        "Code sexe": "sex",
        "Date de naissance": "birth_date",
        "Code de la catégorie socio-professionnelle": "job_code",
    })

    df = df.drop_duplicates()
    log.append(("after removing exact duplicates", len(df)))

    # missing by design: these territories are recorded under a collectivité
    df["territory"] = df["dept"].fillna(df["coll"])

    df["birth_date"] = pd.to_datetime(df["birth_date"])
    df["age"] = (pd.Timestamp("2026-11-02") - df["birth_date"]).dt.days / 365.25
    df["is_woman"] = df["sex"] == "F"

    for step, rows in log:
        print(f"{step:35} {rows:>7,} rows")
    return df


clean_mayors = clean_officials("data/elus_maires_2026.csv")

In [ ]:
print("Women among mayors:", round(clean_mayors["is_woman"].mean() * 100, 1), "%")
print("Median age, by sex:")
print(clean_mayors.groupby("sex")["age"].median().round(1))

by_territory = clean_mayors.groupby("territory")["is_woman"].agg(["mean", "size"])
by_territory["mean"] = (by_territory["mean"] * 100).round(1)
print(by_territory.sort_values("mean", ascending=False).head(4))

**Read the table before quoting it.** Hauts-de-Seine comes first with only 35 communes, so a single mayor moves its rate by about 3 points (1/35), while Ardèche has 334 communes. That is why the table keeps `size` next to `mean`. Always report the number of observations behind a percentage.

### 8.8 On your own: the same function on a new file

##### Exercise 10

`clean_officials` should also work on `data/elus_deputes_2026.csv`. Apply it, then print:
1. the share of women among deputies, rounded to 1 decimal;
2. the number of deputies with a missing `dept`, and the number with a missing `territory`. Did our fix for missing départements also work on this file?
3. the World Bank value for France in `"2025"`, rounded to 1 decimal. Why might it differ from your figure, when both describe women in the Assemblée nationale? Answer in a comment.

In [ ]:
# Write your solution here

Expected output:

```text
raw file                                577 rows
after removing exact duplicates         577 rows
38.3
8 0
36.2
```

##### For your project: start your cleaning log now

Open your own dataset (or a candidate) and fill in this table at the top of a new notebook. Together with the source link and download date, this is the "data" part of your final project.

| Question | Your answer |
|---|---|
| Source link and download date | |
| What is one row? | |
| Which column (or columns) identifies a row? Is it unique? | |
| Rows at the start / rows at the end | |
| Missing values: how many, in which columns, and why? | |
| Which types did you change? | |
| What did you remove, and why? | |